# Lesson 5: A real job

**Project:** Agentic Engineering Worklog
**Model:** `qwen3:8b`, running on this PC through Ollama

Lessons 1 to 4 built the agent one piece at a time inside notebooks. This lesson moves those pieces into a small **program** you can run every day from a terminal, logging to your real work log. It also adds the test lesson 4 showed was missing: a **control**, which shows which tests really depend on memory.

## Where lesson 4 left off

Lesson 4 gave the agent conversation memory and a tool for correcting the last entry, and it scored 6 of 6. But two things came out of it:

1. **The log file often did the job memory was expected to do.** "Actually, make that two hours" worked even with no memory, because the correction tool finds the last entry in the file.
2. **Most of the tests didn't actually test memory.** They could pass without it. The fix is to run the same tests with memory switched off and compare.

There was also a practical problem: everything lived in notebooks. To log real work, you'd have to open a notebook and run a dozen cells first.

## The big ideas: modules, packages and programs

**A module** is a Python file whose code other files can use. Writing `import json` loads Python's own `json` module.

**A package** is a folder of modules that belong together. This lesson's package is the `worklog` folder in this project:

| File | Its one job | Built in |
|---|---|---|
| `config.py` | Settings you might change: the model, project names, limits | lessons 1 to 4 |
| `tools.py` | The tools (save, search, correct) and their descriptions | lessons 2 to 4 |
| `agent.py` | The loop and the conversation memory | lessons 3 and 4 |
| `__main__.py` | The terminal program | this lesson |
| `__init__.py` | Marks the folder as a package, and explains it | this lesson |

**Why split it up?** Each file has one job, so it's easy to find things and change one part without breaking another. The notebook and the terminal program can also share the same code. Fix a bug once in `tools.py`, and both get the fix.

**A program** is code you start from the terminal. `python -m worklog` means "run the `worklog` package as a program". Python finds the package, then runs its `__main__.py` file.

## What we are looking for

By the end of this notebook you should be able to say yes to five things:

1. I know what's in each file of the `worklog` package, and which lesson it came from.
2. I can use the package from a notebook with `import`.
3. I can run the agent as a program from the terminal.
4. I can run a test with a control, and tell which tests really depend on memory.
5. I can log my real work with it, day to day.

**How to run it in PyCharm:** click a code cell and press **Shift + Enter**. Run the cells in order from top to bottom.

**If you change a file in the `worklog` package,** restart the notebook before running it again (the **Restart Kernel** button at the top of the notebook). Python loads a package once and keeps that copy, so edits made after the first `import` aren't picked up until a restart.

## Step 1: Setup

### What we are looking for

The notebook importing the `worklog` package, and pointing it at a demo log, so nothing in this lesson touches your real work log.

In [ ]:
import json
import os
import re
import subprocess
import sys
from datetime import date, timedelta
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from worklog import config, tools
from worklog.agent import Conversation

DEMO_LOG = config.DATA_DIR / "demo_log_lesson5.jsonl"
tools.set_log_file(DEMO_LOG)

TODAY_S = date.today().isoformat()
YESTERDAY = (date.today() - timedelta(days=1)).isoformat()

STARTING_ENTRIES = [
    {"logged_at": f"{YESTERDAY}T09:00:00", "project": "Cornerpost Watch", "description": "Tuned the cloud mask", "hours": 1},
    {"logged_at": f"{YESTERDAY}T14:00:00", "project": "Agentic Engineering Worklog", "description": "Ran lesson 4", "hours": None},
    {"logged_at": f"{TODAY_S}T08:30:00", "project": "Cornerpost Watch", "description": "Fixed the alert email bug", "hours": 2},
    {"logged_at": f"{TODAY_S}T10:00:00", "project": "Herd Planner", "description": "Added vaccine records", "hours": 1.5},
]


def reset_log():
    """Put the demo log back to the four starting entries."""
    tools.write_entries(STARTING_ENTRIES)


reset_log()
print("Package folder:", Path(config.__file__).parent)
print("Model:         ", config.MODEL)
print("Real log:      ", config.WORK_LOG)
print("Using:         ", tools.get_log_file())
print("Entries:       ", len(tools.read_entries()))

### The code, line by line

- The imports are tools used later: `subprocess` runs a program from Python (Step 4), `os` reads the computer's settings for programs, and `sys` gives access to Python's own settings.
- `project_root` is the project folder, worked out as in earlier lessons.
- `sys.path.insert(0, str(project_root))` tells Python to look in the project folder when importing. `sys.path` is the list of folders Python searches for modules. The notebook runs from the `notebooks` folder, so without this line Python wouldn't find `worklog`.
- `from worklog import config, tools` loads two modules from the package. After this, `config.MODEL` means "the `MODEL` setting in `config.py`".
- `from worklog.agent import Conversation` loads the `Conversation` class from `agent.py`: lesson 4's memory, now in the package.
- `tools.set_log_file(DEMO_LOG)` points every tool at a demo log. This is the package's version of lesson 4's `LOG_FILE = DEMO_LOG`.
- `STARTING_ENTRIES` and `reset_log()` are lesson 4's four known entries, so every result here can be checked by hand. `reset_log()` uses the package's own `write_entries` tool.
- The `print` lines show where everything is.

### How to read it

You should see:

- **Package folder:** ending in `Agentic-Engineering-Worklog\worklog`.
- **Model:** `qwen3:8b`.
- **Real log:** ending in `data\work_log.jsonl`. That's the log the terminal program uses.
- **Using:** ending in `data\demo_log_lesson5.jsonl`. This notebook uses the demo log instead.
- **Entries: 4.**

### What it means

The notebook has no agent code of its own any more. It *uses* the package, the same way the terminal program will. That's the point of a package: one copy of the code, used in many places.

### What's next and why

Before using the package, Step 2 takes a quick tour of what's in it.

## Step 2: A tour of the package

### What we are looking for

Each file in `worklog`, how long it is, and the first line of its own description.

In [ ]:
package_dir = Path(config.__file__).parent

for path in sorted(package_dir.glob("*.py")):
    text = path.read_text(encoding="utf-8")
    lines = len(text.splitlines())
    first_line = text.split('"""')[1].strip().splitlines()[0] if '"""' in text else ""
    print(f"{path.name:<14} {lines:>4} lines   {first_line}")

### The code, line by line

- `package_dir` is the `worklog` folder: the folder that holds `config.py`.
- `package_dir.glob("*.py")` finds every file ending in `.py`. `sorted` puts them in alphabetical order.
- For each file:
    - `read_text` reads the whole file.
    - `len(text.splitlines())` counts its lines.
    - `first_line` takes the text between the first pair of triple quotes, which is the file's **docstring**: its own description of itself. Only the first line is kept.
- The `print` lines up the columns: `<14` pads the name to 14 characters, and `>4` right-aligns the line count in 4.

### How to read it

One row per file, about 300 lines in total. The descriptions match the table at the top of this notebook:

- `agent.py` is the loop and the memory.
- `config.py` holds the settings.
- `tools.py` holds the tools, and is the longest, because every tool has both code and a description.
- `__main__.py` is the terminal program.

### What it means

The whole agent fits in about 300 lines, and much of that is comments and descriptions. Everything from lessons 1 to 4 is here, organized by job.

**Open `worklog\config.py` in PyCharm now.** It's short, and it's the file you're most likely to change: your project names, the model, and how much the agent remembers. Every setting has a comment saying what it's for.

### What's next and why

Step 3 checks that the package works the same as lesson 4's notebook code.

## Step 3: Use the package from Python

### What we are looking for

The same correction as lesson 4, handled by the package's `Conversation`, with each tool step printed.

In [ ]:
reset_log()
chat = Conversation(verbose=True)

for text in ["Log an hour on Herd Planner fixing the weight chart.", "Actually, make that two hours."]:
    print("You:  ", text)
    reply, calls = chat.send(text)
    print("Agent:", reply)
    print()

print("Last entry:", tools.read_entries()[-1])

### The code, line by line

- `reset_log()` restores the four starting entries.
- `Conversation(verbose=True)` creates a conversation that prints each tool step. Its memory limit comes from `config.MAX_TURNS`, which is 5.
- The loop sends two messages, as in lesson 4.
- The last line prints the newest entry in the demo log.

### How to read it

You should see an `add_entry` step for the first message, an `update_last_entry` step with `hours: 2` for the second, and a last entry for Herd Planner with **2** hours.

### What it means

This is lesson 4's result, from the package instead of notebook code. Moving the code into files didn't change how it behaves, which is what you want from a move like this.

### What's next and why

Step 4 runs the agent the way you'll actually use it: as a program.

## Step 4: Run the agent as a program

### What we are looking for

The terminal program, `python -m worklog`, handling one message from start to finish, using the demo log.

In [ ]:
def run_program(message):
    """Run `python -m worklog --once ...` against the demo log, and print what it shows."""
    command = [sys.executable, "-m", "worklog", "--log", str(DEMO_LOG), "--once", message]
    env = {**os.environ, "PYTHONIOENCODING": "utf-8"}
    result = subprocess.run(command, cwd=project_root, env=env, capture_output=True, text=True, encoding="utf-8")
    print(result.stdout.strip())
    if result.returncode != 0:
        print(result.stderr.strip())


reset_log()
print("> python -m worklog --once \"Spent 30 minutes on the Cornerpost Logic company site.\"")
run_program("Spent 30 minutes on the Cornerpost Logic company site.")
print()
print("> python -m worklog --once \"How many hours have I logged today?\"")
run_program("How many hours have I logged today?")

### The code, line by line

- `run_program(message)` starts the program, exactly as you would in a terminal:
    - `sys.executable` is the Python program running this notebook, which is this project's `.venv` Python.
    - `"-m", "worklog"` means "run the `worklog` package".
    - `"--log", str(DEMO_LOG)` points it at the demo log. `"--once", message` gives it one message, so it answers and exits instead of waiting for you to type.
    - `env` is a copy of the computer's settings for programs, plus one: `PYTHONIOENCODING` set to `utf-8`. Without it, on Windows, a reply containing a special character such as an em dash could crash the program when its output is collected.
    - `subprocess.run(...)` starts the program and waits for it to finish. `cwd=project_root` runs it from the project folder, and `capture_output=True` collects what it prints so the notebook can show it.
- `print(result.stdout.strip())` shows what the program printed. If it ended with an error (`returncode` isn't 0), the error message is shown too.
- The program is run twice: once to log work, and once to ask a question.

### How to read it

For each run, the command it ran (after `>`), then what the program printed: a `[step 1]` line for each tool it used, then `Agent:` and its reply.

- The first should save **0.5 hours** for Cornerpost Logic.
- The second should search today's entries. The total should be **4 hours**: the starting 2 + 1.5, plus the 0.5 just saved.

### What it means

The two runs were **separate programs**, so the second one had no conversation memory of the first. It still got the right total, because the first run's entry was saved in the log file, and the second run searched it. That's lesson 4's finding again: saved data lasts between sessions; conversation memory doesn't.

### What's next and why

Step 5 measures exactly when conversation memory matters, using a control.

## Step 5: A test with a control

### What we are looking for

For each test conversation, whether it passes **with** memory (`max_turns=5`) and **without** it (`max_turns=0`). Comparing the two shows which tests really depend on memory.

This is the same idea as a control group in data science: to know what one ingredient does, compare against a version without it.

In [ ]:
TEST_CONVERSATIONS = [
    {"name": "Correction of hours",
     "messages": ["Log an hour on Herd Planner fixing the weight chart.", "Actually, make that two hours."],
     "new": [("Herd Planner", 2)], "answer": []},
    {"name": "Main project, not the first in the list",
     "messages": ["My main project this week is Herd Planner.", "Log 3 hours on my main project."],
     "new": [("Herd Planner", 3)], "answer": []},
    {"name": "Project named in an earlier message",
     "messages": ["Everything I log next is for Cornerpost Logic.", "Spent 2 hours on the company site."],
     "new": [("Cornerpost Logic", 2)], "answer": []},
    {"name": "Follow-up question",
     "messages": ["What is the capital of Colorado?", "And what about Wyoming?"],
     "new": [], "answer": [r"cheyenne"]},
    {"name": "Question about what was just said",
     "messages": ["Log 30 minutes on Cornerpost Watch checking alerts.", "How many minutes did I just say?"],
     "new": [("Cornerpost Watch", 0.5)], "answer": [r"\b30\b"]},
    {"name": "Two separate entries",
     "messages": ["Log 2 hours on Cornerpost Watch writing alert tests.", "Also log an hour on Herd Planner."],
     "new": [("Cornerpost Watch", 2), ("Herd Planner", 1)], "answer": []},
]


def same_hours(got, expected):
    try:
        return float(got) == float(expected)
    except (TypeError, ValueError):
        return False


def passes(case, max_turns):
    """Run one test conversation with this much memory. Returns (passed, new entries, last reply)."""
    reset_log()
    chat = Conversation(max_turns=max_turns)
    reply = ""
    for text in case["messages"]:
        reply, _ = chat.send(text)
    entries = tools.read_entries()
    start_ok = entries[:len(STARTING_ENTRIES)] == STARTING_ENTRIES
    new = [(e["project"], e["hours"]) for e in entries[len(STARTING_ENTRIES):]]
    new_ok = len(new) == len(case["new"]) and all(
        g[0] == w[0] and same_hours(g[1], w[1]) for g, w in zip(new, case["new"])
    )
    answer_ok = all(re.search(p, reply or "", re.IGNORECASE) for p in case["answer"])
    return start_ok and new_ok and answer_ok, new, reply


results = []
for case in TEST_CONVERSATIONS:
    with_memory, new_with, reply_with = passes(case, max_turns=5)
    without_memory, new_without, reply_without = passes(case, max_turns=0)
    if with_memory and not without_memory:
        verdict = "needs memory"
    elif with_memory and without_memory:
        verdict = "passes either way"
    elif not with_memory and not without_memory:
        verdict = "fails either way"
    else:
        verdict = "passes only WITHOUT memory"
    results.append((case["name"], with_memory, without_memory, verdict))
    print(case["name"])
    print(f"   with memory:    {'pass' if with_memory else 'FAIL'}  {new_with}  reply: {(reply_with or '').strip()[:90]}")
    print(f"   without memory: {'pass' if without_memory else 'FAIL'}  {new_without}  reply: {(reply_without or '').strip()[:90]}")
    print()

reset_log()
print(f"{'Test':<42} {'With':<6} {'Without':<8} Verdict")
for name, w, wo, verdict in results:
    print(f"{name:<42} {'pass' if w else 'FAIL':<6} {'pass' if wo else 'FAIL':<8} {verdict}")
print()
print("Passed with memory:   ", sum(r[1] for r in results), "of", len(results))
print("Passed without memory:", sum(r[2] for r in results), "of", len(results))
print("Really need memory:   ", sum(r[3] == "needs memory" for r in results))

### The code, line by line

- `TEST_CONVERSATIONS` has six two-message conversations. Each has a `name`, the `messages`, the entries that should be added (`new`), and patterns the last reply must contain (`answer`). Three are new, designed so the model **can't guess** the answer without memory:
    - **"Main project, not the first in the list":** lesson 4's version used Cornerpost Watch, which the model guessed correctly with no memory. This one uses Herd Planner, which it has no reason to guess.
    - **"Project named in an earlier message":** "the company site" doesn't say which project. Only the first message does.
    - **"Question about what was just said":** only the conversation knows you said "30 minutes". The log stores 0.5 hours.
- `passes(case, max_turns)` runs one conversation with a given amount of memory, then checks lesson 4's three things: the starting entries are unchanged, the new entries are right, and the reply contains the right facts.
- The main loop runs every conversation **twice**, with `max_turns=5` and then `max_turns=0`, and gives each test a verdict:
    - **needs memory:** passes with memory, fails without. This is a real test of memory.
    - **passes either way:** memory makes no difference. It's testing something else, such as the tools.
    - **fails either way:** a problem that memory doesn't fix.
    - **passes only WITHOUT memory:** unexpected, and worth a close look.
- At the end, a summary table and three counts.

### How to read it

First a block per test, showing the result, the new entries and the start of the reply, with and without memory. Then the summary table. The expected pattern:

| Test | Expected with memory | Expected without | Expected verdict |
|---|---|---|---|
| Correction of hours | pass | pass | passes either way (lesson 4's finding) |
| Main project, not the first in the list | pass | FAIL | needs memory |
| Project named in an earlier message | pass | FAIL | needs memory |
| Follow-up question | pass | FAIL | needs memory |
| Question about what was just said | pass | probably FAIL | needs memory |
| Two separate entries | pass | pass | passes either way |

If your results differ, that's a finding in itself, as it was in lesson 4. Read the replies to see why.

### What it means

The **with memory** count says how well the agent works. The **without memory** count says how much of that comes from memory. Only the difference between them measures memory itself.

This is how a test should be designed whenever you add a feature: first check the feature helps, then check the test would notice if the feature were gone.

### What's next and why

The agent is tested. Step 6 is how to use it for real.

## Step 6: Use it every day

There's nothing to run in this step. These are the commands to type yourself.

### Start the agent

Open the **Terminal** tab at the bottom of PyCharm, in this project, and run:

```powershell
.venv\Scripts\activate
python -m worklog
```

The first line switches to this project's Python. The second starts the agent. Ollama must be running; if it isn't, the program says so.

### What a session looks like

```text
Work log agent (qwen3:8b). Log: B:\Development\Python\Agentic-Engineering-Worklog\data\work_log.jsonl
Tell me what you worked on, or ask about past work. Type "quit" to stop.

You: Spent 2 hours on Herd Planner fixing the calving chart.
  [step 1] add_entry({...}) -> Saved entry for Herd Planner.
Agent: Saved 2 hours on Herd Planner for fixing the calving chart.

You: How many hours have I logged this week?
  [step 1] search_entries({...}) -> ...
Agent: ...

You: quit
```

Your words follow `You:`, each tool the agent used is shown in `[step ...]` lines, and its answer follows `Agent:`.

### Other ways to run it

| Command | What it does |
|---|---|
| `python -m worklog` | Chat, with tool steps shown |
| `python -m worklog --quiet` | Chat, showing only the replies |
| `python -m worklog --once "Spent an hour on Herd Planner."` | Handle one message and exit: handy for quick logging |
| `python -m worklog --help` | List every option |

### Adding your own projects

Open `worklog\config.py`, add names to `KNOWN_PROJECTS`, and save. The next time you start the agent, it'll use them.

### Where your entries go

Your real log is `data\work_log.jsonl`, in this project. It's plain text, so you can open it in PyCharm any time to see exactly what the agent saved. The `data` folder is in `.gitignore`, so your log is never uploaded to GitHub.

## Exercises

1. **Log a real day.** Use `python -m worklog` for one day of real work. At the end, ask it "What did I work on today?" and check the answer against the log file.
2. **Add a project.** Add a project name to `config.py`, restart the notebook, and add a test to Step 5 that uses it.
3. **Change the memory.** In `config.py`, set `MAX_TURNS = 1`. Restart the notebook and rerun Step 5. Which tests now fail with memory on?
4. **Design a test that needs memory.** Write one more conversation for Step 5 that you think will fail without memory, and check whether it does.

## What it all means

- **A package keeps one copy of the code,** used by both the notebook and the program. Each file has one job.
- **`python -m worklog` runs the agent as a program,** so logging work no longer means opening a notebook.
- **Saved data lasts between sessions; conversation memory doesn't.** Two separate runs in Step 4 still shared the log.
- **A control shows what a test really measures.** Running each test with memory on and off separates "the agent works" from "memory is why it works".

## Words used in this lesson

| Word | Meaning |
|---|---|
| Module | A Python file whose code other files can use with `import`. |
| Package | A folder of modules that belong together, such as `worklog`. |
| `import` | Load a module or package so its code can be used. |
| `sys.path` | The list of folders Python searches when importing. |
| Docstring | A description in triple quotes at the top of a file or function. |
| Program | Code started from the terminal, here with `python -m worklog`. |
| `__main__.py` | The file Python runs when a package is started with `-m`. |
| Command-line option | Extra instructions given when starting a program, such as `--quiet`. |
| `subprocess` | A Python module for running another program and collecting its output. |
| Control | The same test with one thing switched off, to see what that thing really does. |
| Restart the kernel | Start the notebook's Python fresh, so it picks up edited files. |

## What's next and why

All five lessons are done: the work log agent exists, is tested, and runs as a program.

**Next is the presentation:** a short deck telling the story of the project, from the first model call to a working, tested agent, built from these notebooks and their results. It'll be planned first in `docs\PRESENTATION_PLAN.md` for you to review, then built.

After that, the agent is the base for two larger pieces of work: the productivity desktop app, and agent features for Cornerpost Logic and Cornerpost Watch.